# RQ3: How does advertising spend relate to sales revenue across different campaign types, and is there a diminishing returns effect?
**Dataset:** Marketing and Product Performance Dataset (Kaggle)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150
from sklearn.preprocessing import PolynomialFeatures, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')
print('Ready.')

In [ ]:
# ── LOAD DATASET ──────────────────────────────────────────────────────────────
df = pd.read_csv('marketing_and_product_performance.csv')
df = df.dropna(subset=['Revenue_Generated'])

# Identify column names (flexible)
ad_col = 'Budget'
rev_col = 'Revenue_Generated'
campaign_col = [c for c in df.columns if 'campaign' in c.lower()][0] if any('campaign' in c.lower() for c in df.columns) else None

print(f'Ad column: {ad_col}, Campaign column: {campaign_col}')
df[[ad_col, rev_col]].describe()

In [ ]:
# ── FIGURE 3.1 — Scatter: Ad Spend vs Revenue by Campaign Type ────────────────
fig, ax = plt.subplots(figsize=(12, 7))

if campaign_col:
    campaigns = df[campaign_col].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(campaigns)))
    for camp, col in zip(campaigns, colors):
        sub = df[df[campaign_col] == camp]
        ax.scatter(sub[ad_col], sub[rev_col], label=str(camp), alpha=0.5, s=20, color=col)
    ax.legend(title='Campaign Type', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
else:
    ax.scatter(df[ad_col], df[rev_col], alpha=0.4, s=15, color='#2E4057')

ax.set_xlabel(ad_col, fontsize=12)
ax.set_ylabel(rev_col, fontsize=12)
ax.set_title('Figure 3.1 — Ad Spend vs Sales Revenue by Campaign Type (RQ3)', fontweight='bold')
plt.tight_layout()
plt.savefig('Figure_3_1_AdSpend_vs_Revenue_Scatter.pdf', bbox_inches='tight')
plt.show()
print('Figure 3.1 saved.')

In [ ]:
# ── FIGURE 3.2 — Polynomial Regression: Diminishing Returns ──────────────────
df_sorted = df[[ad_col, rev_col]].dropna().sort_values(ad_col)
X_ad = df_sorted[[ad_col]].values
y_rev = df_sorted[rev_col].values

poly = Pipeline([('poly', PolynomialFeatures(degree=2)), ('lr', LinearRegression())])
poly.fit(X_ad, y_rev)
y_poly_pred = poly.predict(X_ad)

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(X_ad, y_rev, alpha=0.3, s=15, color='#CAF0F8', label='Actual Data')
ax.plot(X_ad, y_poly_pred, color='#E07A5F', linewidth=2.5, label='Polynomial Fit (degree=2)')
ax.set_xlabel(ad_col, fontsize=12)
ax.set_ylabel(rev_col, fontsize=12)
ax.set_title('Figure 3.2 — Diminishing Returns: Polynomial Regression Curve (RQ3)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('Figure_3_2_Diminishing_Returns_Polynomial.pdf', bbox_inches='tight')
plt.show()
print('Figure 3.2 saved.')

In [ ]:
# ── TABLE 3.1 — Mean Revenue by Campaign Type and Spend Quartile ──────────────
df['Spend_Quartile'] = pd.qcut(df[ad_col], q=4, labels=['Q1 (Low)', 'Q2', 'Q3', 'Q4 (High)'])

if campaign_col:
    table_31 = df.groupby([campaign_col, 'Spend_Quartile'])[rev_col].agg(['mean', 'median', 'count']).reset_index()
    table_31.columns = ['Campaign_Type', 'Spend_Quartile', 'Mean_Revenue', 'Median_Revenue', 'Count']
else:
    table_31 = df.groupby('Spend_Quartile')[rev_col].agg(['mean', 'median', 'count']).reset_index()
    table_31.columns = ['Spend_Quartile', 'Mean_Revenue', 'Median_Revenue', 'Count']

table_31 = table_31.round(2)
table_31.to_csv('Table_3_1_Revenue_by_Campaign_Spend.csv', index=False)
print('Table 3.1 saved.')
table_31

In [ ]:
# ── FIGURE 3.3 — Mean Revenue per Spend Quartile by Campaign ─────────────────
if campaign_col:
    pivot = table_31.pivot(index='Spend_Quartile', columns='Campaign_Type', values='Mean_Revenue')
    fig, ax = plt.subplots(figsize=(12, 6))
    pivot.plot(kind='bar', ax=ax, colormap='tab10')
    ax.set_xlabel('Ad Spend Quartile')
    ax.set_ylabel('Mean Sales Revenue')
    ax.set_title('Figure 3.3 — Mean Revenue per Spend Quartile by Campaign Type (RQ3)', fontweight='bold')
    ax.legend(title='Campaign Type', bbox_to_anchor=(1.01, 1))
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('Figure_3_3_Revenue_by_Spend_Campaign.pdf', bbox_inches='tight')
    plt.show()
    print('Figure 3.3 saved.')

## RQ3 Summary
The polynomial regression curve (Figure 3.2) reveals whether a diminishing returns pattern exists — i.e., revenue growth slows at higher ad spend levels. Table 3.1 shows how different campaign types respond to increased spend across quartiles.